# Random Forest Regression with Cross-Validation

This demonstration builds toward interrogating a **Random Forest regressor** for experimental band gaps. The trust question throughout is: **how much of the model's apparent quality—and of our explanation of it—depends on choices we made about validation, model selection, and perturbation?**

The expensive experiment suite has already been run. Controls below query a portable SQLite database; they do not silently train new forests.


## Questions to Guide Your Exploration

- Does the model appear good because of the validation strategy we chose?
- How much does a reported score change when the cross-validation partition changes but the forest seed does not?
- Is there one uniquely defensible hyperparameter combination, or a broad plateau of similar evidence?
- Do impurity and permutation importance tell the same explanatory story? Which question does each method answer?
- Can predictive performance remain robust while the explanation changes, or vice versa?
- Which large errors correspond to materials that deserve domain-specific scrutiny?

High scores are evidence about a particular experiment—not proof that a model is trustworthy for every use.


## 1. Notebook Scaffold

The notebook locates the repository whether the kernel starts at the repository root or inside `demonstrations/`. It identifies both the source CSV and the precomputed SQLite file explicitly.


In [ ]:
from pathlib import Path

def find_repository_root(start_directory):
    start_directory = Path(start_directory).resolve()
    for candidate in [start_directory, *start_directory.parents]:
        if (candidate / "demonstrations" / "random_forest_cv.ipynb").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository containing demonstrations/random_forest_cv.ipynb"
    )

REPO_ROOT = find_repository_root(Path.cwd())
DATA_PATH = REPO_ROOT / "demonstrations" / "data" / "featurized_matbench_expt_gap.csv"
DATABASE_PATH = REPO_ROOT / "demonstrations" / "precomputed" / "rfr_precomputed_results.sqlite"
assert DATA_PATH.exists(), f"Missing dataset: {DATA_PATH}"
assert DATABASE_PATH.exists(), (
    f"Missing precomputed results: {DATABASE_PATH}\n"
    "Run: python demonstrations/generate_rfr_precomputed.py --mode smoke"
)
print(f"Repository root: {REPO_ROOT}")
print(f"Dataset:         {DATA_PATH}")
print(f"Precomputed DB:  {DATABASE_PATH}")


## 2. Environment Setup and Imports

Version printouts make the computational environment part of the evidence. Sections 1–10 retrieve costly model results from SQLite. The Random Forest training class is used only by Section 11's explicit one-model training button.


In [ ]:
import hashlib
import json
import sqlite3
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import clear_output, display
from ipywidgets import Button, Dropdown, FloatSlider, HBox, IntSlider, Output, SelectionSlider, VBox, interactive_output

COLORS = {"kfold": "#4c78a8", "loco": "#f58518"}
CV_LABELS = {"kfold": "Shuffled K-fold", "loco": "Feature-space LOCO"}

def query(sql, parameters=()):
    # SQLite binds NumPy integers as BLOBs rather than SQL integers.
    parameters = tuple(value.item() if isinstance(value, np.generic) else value
                       for value in parameters)
    with sqlite3.connect(DATABASE_PATH) as connection:
        return pd.read_sql_query(sql, connection, params=parameters)

metadata_rows = query("SELECT key, value FROM metadata")
METADATA = {row.key: json.loads(row.value) for row in metadata_rows.itertuples()}
current_dataset_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if METADATA.get("schema_version") != 3 or METADATA.get("dataset_sha256") != current_dataset_sha256:
    raise RuntimeError(
        "The precomputed database is incompatible with this notebook or source CSV. "
        "Regenerate it with --overwrite before continuing."
    )
print({
    "python": sys.version.split()[0], "numpy": np.__version__,
    "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
    "database_mode": METADATA["mode"], "schema_version": METADATA["schema_version"],
})

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


## 3. Dataset Loading and Inspection

The file contains one domain-context field, one response, and numeric composition-derived descriptors. We identify the target by its experimental band-gap name rather than assuming its position.

`composition` is preserved so an unusual point can be reconnected to a real material. It is **not** a feature, target, clustering variable, or projection variable. The Random Forest receives the remaining raw numeric columns: no polynomial expansion and no standardization. This dataset has no missing numeric values and no constant features. The generator deliberately fails fast if numeric values are missing: any future imputation must learn its values inside each training fold, never from the full dataset.


In [ ]:
df = pd.read_csv(DATA_PATH)
target_candidates = [
    name for name in df.columns
    if "gap" in name.lower()
    and any(token in name.lower() for token in ("expt", "experiment", "target"))
    and pd.api.types.is_numeric_dtype(df[name])
]
assert len(target_candidates) == 1, f"Ambiguous target candidates: {target_candidates}"
target_column = target_candidates[0]

compositions = df["composition"].astype(str).copy()
y = df[target_column].astype(float).copy()
numeric_candidates = [
    name for name in df.select_dtypes(include=np.number).columns
    if name != target_column
]
constant_columns = [name for name in numeric_candidates if df[name].nunique(dropna=True) <= 1]
feature_columns = [name for name in numeric_candidates if name not in constant_columns]
X = df[feature_columns].copy()  # raw numeric values; no scaling for Random Forest training
nonnumeric_columns = [name for name in df.columns if not pd.api.types.is_numeric_dtype(df[name])]

if X.isna().any().any():
    missing_columns = X.columns[X.isna().any()].tolist()
    raise ValueError(
        f"Missing numeric feature values in {missing_columns}. Do not impute from the full dataset; "
        "any imputation must be learned within each training fold."
    )
if y.isna().any():
    raise ValueError("The target contains missing values and requires an explicit data decision.")

assert "composition" not in X.columns
assert target_column not in X.columns
assert list(X.columns) == METADATA["feature_columns"]

inspection = pd.Series({
    "observations": len(df), "columns in file": df.shape[1],
    "target": target_column, "usable numeric features": X.shape[1],
    "missing feature values": int(X.isna().sum().sum()),
    "constant numeric columns excluded": len(constant_columns),
    "nonnumeric columns": ", ".join(nonnumeric_columns),
}, name="dataset audit")
display(inspection.to_frame())
display(pd.DataFrame({"composition": compositions.head(6), "measured band gap (eV)": y.head(6)}))

with plt.ioff():
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(y, bins=35, color="#4c78a8", alpha=0.85)
axes[0].set(title="Experimental band-gap distribution", xlabel="measured band gap (eV)", ylabel="observations")
feature_summary = X.agg(["min", "median", "max"]).T
axes[1].hist(np.log10(feature_summary["max"] - feature_summary["min"] + 1e-12), bins=30,
             color="#54a24b", alpha=0.85)
axes[1].set(title="Feature ranges span many scales", xlabel="log10(maximum - minimum)", ylabel="features")
figure.tight_layout()
display(figure)
plt.close(figure)


## 4. Random Forest Regression and Ensemble Size

An individual decision tree partitions feature space into regions with similar target values. A Random Forest fits many related but non-identical trees using bootstrap samples and feature subsampling, then averages their predictions.

The number of trees controls how well that average has converged. It is not used as a later model-complexity axis: after performance levels off, additional trees mostly buy a more stable ensemble estimate at greater computational cost. The plot below uses precomputed shuffled K-fold out-of-fold predictions.


In [ ]:
convergence = query("SELECT * FROM convergence_results ORDER BY n_estimators, cv_seed")
summary = convergence.groupby("n_estimators")[["r2", "mae", "rmse"]].agg(["mean", "std"])
fixed_trees = int(METADATA["config"]["fixed_n_estimators"])

with plt.ioff():
    figure, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for axis, metric, label in zip(axes, ["r2", "mae", "rmse"], ["Validation R²", "Validation MAE (eV)", "Validation RMSE (eV)"]):
    grouped = convergence.groupby("n_estimators")[metric]
    mean = grouped.mean()
    std = grouped.std().fillna(0)
    axis.errorbar(mean.index, mean, yerr=std, marker="o", color="#4c78a8", capsize=3)
    axis.axvline(fixed_trees, color="#e45756", linestyle="--", label=f"fixed = {fixed_trees}")
    axis.set(xlabel="number of trees", ylabel=label, title=label)
    axis.grid(color="0.92")
    axis.legend()
figure.suptitle("Does the ensemble estimate appear reasonably converged?")
figure.tight_layout()
display(figure)
plt.close(figure)
print(f"All subsequent precomputed analyses fix n_estimators = {fixed_trees} and RF random_state = {METADATA['rf_seed']}.")


## 5. Cross-Validation Splitters

We compare two kinds of evidence:

- **Shuffled K-fold** holds out randomly mixed observations and is mainly interpolation-like.
- **Leave-one-cluster-out (LOCO)** standardizes the numeric feature matrix only for KMeans, forms coherent regions in the full feature space, and holds out one region at a time. Forests still train on raw features.

For the visualization only, the already-standardized numeric feature space is projected to two principal components. KMeans used all standardized feature dimensions—not these two displayed coordinates—and `composition` entered neither operation.

Cluster construction has seen the complete feature distribution. That modest information leak means LOCO here is a pedagogical generalization **stress test**, not an unbiased estimate of deployment performance. Select a cluster count and connect the geometry back to example compositions.


In [ ]:
cluster_counts = query("SELECT DISTINCT cluster_count FROM cluster_assignments ORDER BY cluster_count")["cluster_count"].tolist()
cluster_count_widget = SelectionSlider(options=cluster_counts, value=cluster_counts[0], description="clusters", continuous_update=False)

def render_clusters(cluster_count):
    points = query(
        """SELECT c.observation_id, c.cluster_id, c.pc1, c.pc2, o.composition, o.observed_target
           FROM cluster_assignments c JOIN observations o USING (observation_id)
           WHERE c.cluster_count=? ORDER BY c.observation_id""", (cluster_count,)
    )
    assert len(points) == len(df)
    with plt.ioff():
        figure, axis = plt.subplots(figsize=(8, 5.5))
    scatter = axis.scatter(points.pc1, points.pc2, c=points.cluster_id, cmap="tab10", s=10, alpha=0.55)
    axis.set(title=f"Full-space KMeans groups shown in a 2-D PCA projection ({cluster_count} clusters)",
             xlabel="principal component 1 (visualization only)", ylabel="principal component 2 (visualization only)")
    colorbar = figure.colorbar(scatter, ax=axis)
    colorbar.set_label("full-space cluster")
    figure.tight_layout()
    display(figure)
    plt.close(figure)
    examples = (points.sort_values(["cluster_id", "observed_target"])
                .groupby("cluster_id", as_index=False).head(3)
                [["cluster_id", "composition", "observed_target"]])
    display(examples.style.format({"observed_target": "{:.3f}"}))

cluster_output = interactive_output(render_clusters, {"cluster_count": cluster_count_widget})
display(VBox([cluster_count_widget, cluster_output]))


## 6. Hyperparameter Selection Before vs. Within the Train–Validation Split

Choose a saved tuning study: **tree depth**, **minimum samples per leaf**, or **minimum samples per split**. Each varies one parameter; the others stay at unlimited depth, leaf size 1, split size 2, and feature fraction 0.5. The forest seed and tree count stay fixed.

- **Same evidence:** choose the candidate with the lowest pooled CV RMSE, then report performance on those same predictions.
- **Nested:** choose the candidate using inner CV inside each outer training fold, then evaluate on the untouched outer fold. Each outer fold can select a different value.

Each grey line joins one outer CV seed's two estimates; red diamonds show their mean. Higher R² is better; lower MAE/RMSE is better. These are pooled out-of-fold metrics, not scores for individual trees. A penalty is not guaranteed for every seed.

Leaf-size candidates run from **1 to 32**; split-size candidates from **2 to 64**. A split needs at least two observations, so scikit-learn does not accept split size 1. The small values allow highly detailed trees; larger values impose stronger restrictions. This does not guarantee overfitting for every dataset.

Controls read stored results without fitting. Adding a candidate set creates a separate selection study, preserving previous comparisons. Only seeds with both workflows complete are shown. To generate additional studies, see the incremental-generation instructions at the end of this notebook.


In [ ]:
selection_tables = set(query("SELECT name FROM sqlite_master WHERE type='table'").name)
if "selection_experiments" in selection_tables:
    selection_studies = query(
        """SELECT DISTINCT e.* FROM selection_experiments e
           JOIN selection_scores s USING (experiment) ORDER BY parameter, candidates"""
    )
else:
    selection_studies = pd.DataFrame()

# Older databases still display their original depth experiment.
selection_options = [("Tree depth (original saved study)", "legacy")]
parameter_labels = {"max_depth": "Tree depth", "min_samples_leaf": "Minimum samples per leaf",
                    "min_samples_split": "Minimum samples per split"}
for study in selection_studies.itertuples():
    values = ["unlimited" if value is None else value for value in json.loads(study.candidates)]
    selection_options.append((f"{parameter_labels[study.parameter]}: {values}", study.experiment))
original_config = METADATA["config"]
migrated_original = (
    not selection_studies.empty
    and ((selection_studies.parameter == "max_depth")
         & (selection_studies.candidates == json.dumps(original_config["selection_depths"]))
         & (selection_studies.outer_folds == original_config["selection_outer_folds"])
         & (selection_studies.inner_folds == original_config["selection_inner_folds"])).any()
)
if migrated_original or query("SELECT COUNT(*) AS n FROM model_selection_summary").n.iloc[0] == 0:
    selection_options = [option for option in selection_options if option[1] != "legacy"]

selection_study_widget = Dropdown(options=selection_options, description="tune", layout={"width": "95%"})

def render_selection_study(experiment):
    if experiment == "legacy":
        selection = query("SELECT * FROM model_selection_summary ORDER BY outer_seed, workflow")
        decisions = query("""SELECT workflow, outer_seed, outer_fold, selected_max_depth AS selected_value
                             FROM model_selection_depth_choices ORDER BY outer_seed, workflow, outer_fold""")
        decisions["selected_value"] = decisions.selected_value.map(lambda value: "unlimited" if value == -1 else value)
        title = "Tree depth (original saved study)"
    else:
        selection = query("SELECT * FROM selection_scores WHERE experiment=? ORDER BY outer_seed, workflow", (experiment,))
        decisions = query("""SELECT workflow, outer_seed, outer_fold, selected_value FROM selection_choices
                             WHERE experiment=? ORDER BY outer_seed, workflow, outer_fold""", (experiment,))
        decisions["selected_value"] = decisions.selected_value.map(
            lambda value: "unlimited" if json.loads(value) is None else json.loads(value))
        title = dict((value, label) for label, value in selection_options)[experiment]
    paired = selection.pivot(index="outer_seed", columns="workflow", values=["r2", "mae", "rmse"]).dropna()
    if paired.empty:
        print("No complete paired seeds are stored yet.")
        return
    selection = selection[selection.outer_seed.isin(paired.index)]
    decisions = decisions[decisions.outer_seed.isin(paired.index)].copy()
    print(f"{title} | {len(paired)} paired outer CV seeds")
    display(selection[["workflow", "outer_seed", "r2", "mae", "rmse"]].round(4))
    decisions["outer_fold"] = decisions.outer_fold.map(lambda value: "all (same evidence)" if value == -1 else value + 1)
    print("Selected values: nested selection makes one decision per outer fold.")
    display(decisions)
    with plt.ioff():
        figure, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for axis, metric, label in zip(axes, ["r2", "mae", "rmse"], ["R²", "MAE (eV)", "RMSE (eV)"]):
        for index, seed in enumerate(paired.index):
            axis.plot([0, 1], [paired.loc[seed, (metric, workflow)] for workflow in ["same_evidence", "nested"]],
                      color="0.7", marker="o", alpha=0.7, label="one outer CV seed" if index == 0 else None)
        axis.plot([0, 1], [paired[(metric, workflow)].mean() for workflow in ["same_evidence", "nested"]],
                  color="#e45756", marker="D", linewidth=2, label="mean across outer CV seeds")
        axis.set(xticks=[0, 1], xticklabels=["same evidence", "nested"], ylabel=label, title=label)
        axis.grid(axis="y", color="0.92")
        axis.legend()
    figure.suptitle(title)
    figure.tight_layout()
    display(figure)
    plt.close(figure)

if selection_options:
    selection_output = interactive_output(render_selection_study, {"experiment": selection_study_widget})
    display(VBox([selection_study_widget, selection_output]))
else:
    print("No saved selection experiments. Run the generator with --studies model_selection first.")


## 7. Validation Metrics Across CV Seeds

A single partition can give an overly tidy view of model quality. These precomputed distributions vary the CV seed while holding the Random Forest seed fixed. Each point is one pooled out-of-fold result; overlapping partitions mean the points diagnose sensitivity to the validation procedure rather than independent repeated experiments.

Use the count control to compare like with like where the database provides multiple counts. The table then aggregates each material's absolute error across **all stored CV seeds** and shows the six largest mean errors per validation method. `composition` makes those recurring statistical failures scientifically discussable without entering the model.


In [ ]:
validation = query("SELECT * FROM validation_results")
count_options = sorted(validation.fold_count.unique())
validation_count_widget = SelectionSlider(options=count_options, value=count_options[0], description="folds/clusters", continuous_update=False)

def render_validation_distributions(fold_count):
    current = validation[validation.fold_count == fold_count]
    with plt.ioff():
        figure, axes = plt.subplots(1, 3, figsize=(15, 5))
    for axis, metric, label in zip(axes, ["r2", "mae", "rmse"], ["Validation R²", "Validation MAE (eV)", "Validation RMSE (eV)"]):
        groups = [current.loc[current.cv_method == method, metric].to_numpy() for method in ["kfold", "loco"]]
        violins = axis.violinplot(groups, positions=[0, 1], showmeans=False, showmedians=True, showextrema=False)
        for body, method in zip(violins["bodies"], ["kfold", "loco"]):
            body.set_facecolor(COLORS[method]); body.set_alpha(0.3)
        for position, (method, values) in enumerate(zip(["kfold", "loco"], groups)):
            jitter = np.linspace(-0.06, 0.06, max(len(values), 1))[:len(values)]
            axis.scatter(position + jitter, values, color=COLORS[method], s=22, alpha=0.7)
            axis.scatter(position, np.mean(values), color="red", marker="D", zorder=4)
        axis.set(xticks=[0, 1], xticklabels=["K-fold", "LOCO"], ylabel=label, title=label)
        axis.grid(axis="y", color="0.92")
    figure.suptitle(f"Dependence on the validation partition | {fold_count} folds/clusters | RF seed fixed")
    figure.tight_layout()
    display(figure)
    plt.close(figure)

    errors = query(
        """SELECT v.cv_method, v.cv_seed, p.observation_id, p.composition,
                  ABS(p.y_true-p.y_pred) AS absolute_error
           FROM validation_predictions p JOIN validation_results v USING (experiment_id)
           WHERE v.fold_count=?""", (int(fold_count),)
    )
    error_summary = (
        errors.groupby(["cv_method", "observation_id", "composition"], as_index=False)
        .agg(mean_absolute_error=("absolute_error", "mean"),
             max_absolute_error=("absolute_error", "max"),
             cv_partitions=("cv_seed", "nunique"))
        .sort_values(["cv_method", "mean_absolute_error"], ascending=[True, False])
        .groupby("cv_method", as_index=False).head(6)
    )
    print("Materials with the largest mean absolute OOF error across stored CV seeds")
    display(error_summary.round(3))

validation_output = interactive_output(render_validation_distributions, {"fold_count": validation_count_widget})
display(VBox([validation_count_widget, validation_output]))


## 8. Exploring GIFTERS: Generalizability

The Random Forest landscape is four-dimensional: `max_depth`, `min_samples_leaf`, `min_samples_split`, and `max_features`. Select any two different parameters for the contour axes; the remaining two become fixed-value sliders. You may also change the validation strategy, fold/cluster count, and metric.

The left panel shows mean performance across precomputed CV seeds. The right shows standard deviation across those seeds. For MAE and RMSE, lower is better; for R², higher is better. Do not reduce this landscape to “find the maximum.” Ask instead:

- Is there one obvious optimum or a broad plateau?
- Does LOCO favor a different complexity region?
- Is the strongest mean evidence also insensitive to partition choice?
- Does the conclusion change when a different pair occupies the axes?


In [ ]:
hyper = query("SELECT * FROM hyperparameter_results")
PARAMETERS = ["max_depth", "min_samples_leaf", "min_samples_split", "max_features"]
PARAMETER_LABELS = {
    "max_depth": "maximum tree depth", "min_samples_leaf": "minimum samples per leaf",
    "min_samples_split": "minimum samples to split", "max_features": "feature fraction per split",
}
def _ordered_parameter_values(name):
    values = hyper[name].unique().tolist()
    if name == "max_depth":
        finite = sorted(value for value in values if value != -1)
        return finite + ([-1] if -1 in values else [])  # unlimited belongs after every finite depth
    return sorted(values)

PARAMETER_VALUES = {name: _ordered_parameter_values(name) for name in PARAMETERS}

x_parameter_widget = Dropdown(options=PARAMETERS, value="max_depth", description="x axis")
y_parameter_widget = Dropdown(options=[p for p in PARAMETERS if p != "max_depth"], value="min_samples_leaf", description="y axis")
cv_method_widget = Dropdown(options=[("Shuffled K-fold", "kfold"), ("Feature-space LOCO", "loco")], value="kfold", description="validation")
hyper_count_widget = SelectionSlider(options=sorted(hyper.fold_count.unique()), value=int(hyper.fold_count.min()), description="folds/clusters", continuous_update=False)
metric_widget = Dropdown(options=[("R² (higher is better)", "r2"), ("MAE (lower is better)", "mae"), ("RMSE (lower is better)", "rmse")], value="r2", description="metric")
fixed_widgets = {
    name: SelectionSlider(options=PARAMETER_VALUES[name], value=PARAMETER_VALUES[name][0],
                          description=PARAMETER_LABELS[name][:18], continuous_update=False)
    for name in PARAMETERS
}
fixed_box = HBox()

def synchronize_axes(change=None):
    allowed_y = [name for name in PARAMETERS if name != x_parameter_widget.value]
    y_parameter_widget.options = allowed_y
    if y_parameter_widget.value not in allowed_y:
        y_parameter_widget.value = allowed_y[0]
    remaining = [name for name in PARAMETERS if name not in {x_parameter_widget.value, y_parameter_widget.value}]
    fixed_box.children = tuple(fixed_widgets[name] for name in remaining)

x_parameter_widget.observe(synchronize_axes, names="value")
y_parameter_widget.observe(synchronize_axes, names="value")
synchronize_axes()

def _axis_labels(name, values):
    return ["unlimited" if name == "max_depth" and value == -1 else f"{value:g}" for value in values]

def render_hyperparameter_map(x_parameter, y_parameter, cv_method, fold_count, metric, **fixed):
    if x_parameter == y_parameter:
        raise ValueError("Choose different parameters for the two axes")
    current = hyper[(hyper.cv_method == cv_method) & (hyper.fold_count == fold_count)].copy()
    for name in PARAMETERS:
        if name not in {x_parameter, y_parameter}:
            current = current[np.isclose(current[name], fixed[name])]
    grouped = current.groupby([y_parameter, x_parameter])[metric].agg(["mean", "std"]).reset_index()
    x_values = PARAMETER_VALUES[x_parameter]
    y_values = PARAMETER_VALUES[y_parameter]
    with plt.ioff():
        figure, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
    for axis, statistic, title in zip(axes, ["mean", "std"], [f"mean validation {metric.upper()}", f"variability across CV seeds (SD of {metric.upper()})"]):
        matrix = grouped.pivot(index=y_parameter, columns=x_parameter, values=statistic).reindex(index=y_values, columns=x_values).to_numpy()
        if np.isnan(matrix).all():
            axis.text(0.5, 0.5, "No precomputed values for this selection", ha="center", va="center")
            continue
        image = axis.contourf(np.arange(len(x_values)), np.arange(len(y_values)), matrix, levels=12, cmap="viridis")
        colorbar = figure.colorbar(image, ax=axis)
        colorbar.set_label(title)
        axis.set(xticks=np.arange(len(x_values)), xticklabels=_axis_labels(x_parameter, x_values),
                 yticks=np.arange(len(y_values)), yticklabels=_axis_labels(y_parameter, y_values),
                 xlabel=PARAMETER_LABELS[x_parameter], ylabel=PARAMETER_LABELS[y_parameter], title=title)
    figure.suptitle(f"{CV_LABELS[cv_method]} | {fold_count} folds/clusters | cached results only")
    display(figure)
    plt.close(figure)

hyper_output = interactive_output(render_hyperparameter_map, {
    "x_parameter": x_parameter_widget, "y_parameter": y_parameter_widget,
    "cv_method": cv_method_widget, "fold_count": hyper_count_widget, "metric": metric_widget,
    **fixed_widgets,
})
display(VBox([HBox([x_parameter_widget, y_parameter_widget]), HBox([cv_method_widget, hyper_count_widget, metric_widget]), fixed_box, hyper_output]))


## 9. Interpretability / Explainability

LASSO coefficients are model parameters that can be inspected directly. A Random Forest has no comparable single coefficient vector, so we turn from intrinsic parameter inspection to **explanation methods**.

- **Impurity importance** summarizes how much tree splits using a feature reduced training impurity.
- **Permutation importance** asks how held-out predictive R² changes when that feature's information is disrupted.

The representative configuration comes from a broad good-performance region, not a claim that one exact tuple is uniquely optimal. Held-out permutation importance is repeated with documented seeds; fold-level distributions are retained. Correlated or redundant descriptors can divide, substitute, or mask importance, so disagreement is evidence about the explanation method—not a bug to hide and not proof that either ranking is “the truth.”

The two bar panels deliberately use the **same union of feature names in the same row order**: the top-*N* impurity features combined with the top-*N* permutation features. A feature such as `MagpieData mean NpValence` therefore appears in both panels even if one method assigns it little importance. The table below focuses on the ten features ranked highest by **at least one** method. It retains both ranks and their gap, but does not elevate weak features merely because their low-importance rankings disagree.


In [ ]:
importance_methods = ["impurity", "permutation"]
importance_method_widget = Dropdown(options=[("Shuffled K-fold", "kfold"), ("Feature-space LOCO", "loco")], value="kfold", description="validation")
top_n_widget = IntSlider(min=5, max=20, step=1, value=12, description="top/method", continuous_update=False)

importance_counts = query("SELECT DISTINCT fold_count FROM importance_runs ORDER BY fold_count").fold_count.tolist()
importance_count_widget = SelectionSlider(options=importance_counts, description="folds/clusters", continuous_update=False)

def render_importance(cv_method, top_n, fold_count):
    values = query(
        """SELECT f.fold, f.feature_name, f.method, f.importance, f.importance_std
           FROM feature_importances f JOIN importance_runs r USING (run_id)
           WHERE r.cv_method=? AND r.fold_count=?""", (cv_method, fold_count)
    )
    means = values.groupby(["method", "feature_name"]).importance.mean().unstack("method").fillna(0)
    ranks = means.rank(method="min", ascending=False)
    shared_features = set(means.nlargest(top_n, "impurity").index) | set(means.nlargest(top_n, "permutation").index)
    # Weakest best-rank first puts the most highly ranked shared features at the top.
    shared_order = ranks.loc[list(shared_features)].min(axis=1).sort_values(ascending=False).index
    display_values = means.loc[shared_order]
    spread = values.groupby(["method", "feature_name"]).importance.std().unstack("method").fillna(0).reindex(display_values.index)
    positions = np.arange(len(display_values))
    with plt.ioff():
        figure, axes = plt.subplots(1, 2, figsize=(15, max(6, 0.35 * len(display_values))))
    for axis, method, color in zip(axes, importance_methods, ["#4c78a8", "#f58518"]):
        axis.barh(positions, display_values[method], xerr=spread[method], color=color, alpha=0.8, capsize=2)
        axis.set(yticks=positions, yticklabels=display_values.index,
                 xlabel=f"{method} importance", title=f"{method.title()} importance\nmean ± fold/seed SD")
        axis.axvline(0, color="0.4", linewidth=0.8)
        axis.grid(axis="x", color="0.92")
    figure.suptitle(f"Two explanatory lenses on the same {len(display_values)} features | {CV_LABELS[cv_method]}")
    figure.tight_layout()
    display(figure)
    plt.close(figure)
    best_rank = ranks.min(axis=1)
    mean_rank = ranks.mean(axis=1)
    top_features = (pd.DataFrame({"best_rank": best_rank, "mean_rank": mean_rank})
                    .sort_values(["best_rank", "mean_rank"]).head(10).index)
    comparison = means.loc[top_features].assign(
        impurity_rank=ranks.loc[top_features, "impurity"],
        permutation_rank=ranks.loc[top_features, "permutation"],
        best_rank=best_rank.loc[top_features],
        absolute_rank_gap=(ranks.loc[top_features, "impurity"] - ranks.loc[top_features, "permutation"]).abs(),
    )
    print("Ten features ranked highest by at least one explanation method:")
    display(comparison.round(4))

importance_output = interactive_output(render_importance, {"cv_method": importance_method_widget, "top_n": top_n_widget, "fold_count": importance_count_widget})
display(VBox([HBox([importance_method_widget, top_n_widget, importance_count_widget]), importance_output]))


## 10. Robustness

### 10.1 Performance Under Controlled Feature and Target Perturbations

For each `(CV method, CV seed, outer fold)`, the generator draws one standard-normal feature matrix $Z_X$ and one standard-normal target vector $Z_y$. It reuses that same noise direction at every displayed magnitude. Feature noise is scaled by each feature's **training-fold standard deviation**; a level of 0.10 therefore means 10% of that local scale. Target noise is similarly scaled by the training-fold target standard deviation. Validation inputs and clean targets remain unchanged, so the question is how increasing the magnitude of one fixed corruption direction changes clean-data performance.

Constant-within-fold features receive no feature noise. `composition` is never perturbed because it is not in the numeric feature matrix. The forest seed stays fixed; perturbation and CV seeds are recorded separately. Because each CV seed defines both a partition and deterministic fold-level noise draws, the right panel is **across-CV-seed variability**, not an estimate based on independent repeated experimental-noise realizations. All results below are cached.


In [ ]:
robustness = query("SELECT * FROM robustness_results")
robust_cv_widget = Dropdown(options=[("Shuffled K-fold", "kfold"), ("Feature-space LOCO", "loco")], value="kfold", description="validation")
robust_metric_widget = Dropdown(options=[("R² (higher is better)", "r2"), ("MAE (lower is better)", "mae"), ("RMSE (lower is better)", "rmse")], value="r2", description="metric")

robust_count_widget = SelectionSlider(options=sorted(robustness.fold_count.unique().tolist()), description="folds/clusters", continuous_update=False)

def render_robustness_performance(cv_method, metric, fold_count):
    current = robustness[(robustness.cv_method == cv_method) & (robustness.fold_count == fold_count)]
    grouped = current.groupby(["target_noise", "feature_noise"])[metric].agg(["mean", "std"])
    feature_levels = sorted(current.feature_noise.unique())
    target_levels = sorted(current.target_noise.unique())
    with plt.ioff():
        figure, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
    for axis, statistic, title in zip(axes, ["mean", "std"], [f"mean validation {metric.upper()}", f"across-seed SD of {metric.upper()}"]):
        matrix = grouped[statistic].unstack("feature_noise").reindex(index=target_levels, columns=feature_levels).to_numpy()
        if np.isnan(matrix).all():
            axis.text(0.5, 0.5, "Requires multiple robustness CV seeds\n(run FULL_PRECOMPUTE)", ha="center", va="center")
            axis.set(xlabel="relative feature noise", ylabel="relative target noise", title=title,
                     xticks=feature_levels, yticks=target_levels)
            continue
        image = axis.contourf(feature_levels, target_levels, matrix, levels=12, cmap="viridis")
        colorbar = figure.colorbar(image, ax=axis); colorbar.set_label(title)
        axis.set(xlabel="relative feature noise", ylabel="relative target noise", title=title,
                 xticks=feature_levels, yticks=target_levels)
    figure.suptitle(f"Robustness of clean-data performance | {CV_LABELS[cv_method]}")
    display(figure)
    plt.close(figure)

robust_performance_output = interactive_output(render_robustness_performance, {"cv_method": robust_cv_widget, "metric": robust_metric_widget, "fold_count": robust_count_widget})
display(VBox([HBox([robust_cv_widget, robust_metric_widget, robust_count_widget]), robust_performance_output]))


### 10.2 Do Explanations and Individual Materials Change Too?

The importance maps summarize the stored CV seeds. The material tables instead compare **clean-training and noisy-training predictions for the same held-out observation**, paired by validation method, CV seed, forest seed, and fold count. The original database stores individual predictions for one representative seed; new runs save every requested seed. Stored seeds are identified below, and each row is a material–seed pair. Training features/targets are perturbed; held-out inputs and experimental targets stay clean.

Both tables rank by **absolute prediction change**, with signed change defined as noisy prediction minus clean prediction. Experimental band gap is the ground truth, not another model prediction. Clean and noisy absolute errors are secondary context. All prediction, change, and error columns use eV.

1. **Largest prediction changes overall:** includes materials that were already poorly predicted.
2. **Initially well-predicted materials:** restricts clean absolute error to the adjustable threshold, then shows the largest prediction changes. The default threshold of 0.25 eV is an illustrative choice, not a scientific accuracy standard. Noise can improve or worsen error; movement alone does not imply degradation.

The fold-count control is shared with Section 10.1. The feature-importance maps summarize the whole noise grid at that fold count; the noise sliders select the two material tables. Return to the compositions to investigate chemistry, descriptor limitations, sparse coverage, or data quality.


In [ ]:
robust_explain_cv_widget = Dropdown(options=[("Shuffled K-fold", "kfold"), ("Feature-space LOCO", "loco")], value="kfold", description="validation")
robust_top_widget = IntSlider(min=3, max=10, step=1, value=6, description="top features", continuous_update=False)
feature_noise_options = sorted(robustness.feature_noise.unique())
target_noise_options = sorted(robustness.target_noise.unique())
example_feature_noise_widget = SelectionSlider(options=feature_noise_options, value=feature_noise_options[-1], description="feature noise", continuous_update=False)
example_target_noise_widget = SelectionSlider(options=target_noise_options, value=target_noise_options[-1], description="target noise", continuous_update=False)

clean_error_limit_widget = FloatSlider(
    min=0.0, max=2.0, step=0.05, value=0.25,
    description="clean error ≤", readout_format=".2f", continuous_update=False,
)

def robustness_prediction_changes(cv_method, feature_noise, target_noise, fold_count):
    pairs = query(
        """SELECT p.observation_id, p.composition, r.cv_seed, r.fold_count,
                  p.y_true_clean AS experimental_gap,
                  b.y_pred AS clean_prediction, p.y_pred AS noisy_prediction
           FROM robustness_predictions p
           JOIN robustness_results r ON r.result_id=p.result_id
           JOIN robustness_results baseline
             ON baseline.cv_method=r.cv_method AND baseline.cv_seed=r.cv_seed
            AND baseline.rf_seed=r.rf_seed AND baseline.fold_count=r.fold_count
            AND baseline.feature_noise=0 AND baseline.target_noise=0
           JOIN robustness_predictions b
             ON b.result_id=baseline.result_id AND b.observation_id=p.observation_id
           WHERE r.cv_method=? AND r.fold_count=? AND ABS(r.feature_noise-?) < 1e-12
             AND ABS(r.target_noise-?) < 1e-12""",
        (cv_method, fold_count, feature_noise, target_noise),
    )
    pairs["prediction_change"] = pairs.noisy_prediction - pairs.clean_prediction
    pairs["absolute_prediction_change"] = pairs.prediction_change.abs()
    pairs["clean_absolute_error"] = (pairs.clean_prediction - pairs.experimental_gap).abs()
    pairs["noisy_absolute_error"] = (pairs.noisy_prediction - pairs.experimental_gap).abs()
    return pairs.sort_values(
        ["absolute_prediction_change", "observation_id", "cv_seed"],
        ascending=[False, True, True],
    )

def render_robustness_explanations(cv_method, top_n, feature_noise, target_noise, clean_error_limit, fold_count):
    importance = query(
        """SELECT r.feature_noise, r.target_noise, r.cv_seed, i.feature_name, i.importance
           FROM robustness_importances i JOIN robustness_results r USING (result_id)
           WHERE r.cv_method=? AND r.fold_count=?""", (cv_method, fold_count)
    )
    if importance.empty:
        print(f"No saved robustness importances for {CV_LABELS[cv_method]} with {fold_count} folds/clusters.")
        return
    baseline = (importance[np.isclose(importance.feature_noise, 0) & np.isclose(importance.target_noise, 0)]
                .groupby("feature_name").importance.mean().nlargest(top_n))
    if baseline.empty:
        print("No zero-noise baseline is saved for this setting. Generate the clean baseline first.")
        return
    top_n = min(top_n, len(baseline))
    selected = importance[importance.feature_name.isin(baseline.index)]
    averaged = selected.groupby(["target_noise", "feature_noise", "feature_name"]).importance.mean().reset_index()
    with plt.ioff():
        figure, axes = plt.subplots(1, top_n, figsize=(3.2 * top_n, 4.5), sharex=True, sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    for axis, feature in zip(axes, baseline.index):
        part = averaged[averaged.feature_name == feature]
        matrix = part.pivot(index="target_noise", columns="feature_noise", values="importance").reindex(
            index=target_noise_options, columns=feature_noise_options
        ).to_numpy()
        image = axis.imshow(matrix, origin="lower", aspect="auto", cmap="magma",
                            extent=[min(feature_noise_options), max(feature_noise_options), min(target_noise_options), max(target_noise_options)])
        axis.set(title=feature.replace("MagpieData ", "", 1)[:24], xlabel="feature noise")
        if axis is axes[0]: axis.set_ylabel("target noise")
        figure.colorbar(image, ax=axis, shrink=0.75, label="mean impurity importance")
    figure.suptitle(f"Can the explanatory ranking move under perturbation? | {CV_LABELS[cv_method]}")
    display(figure)
    plt.close(figure)

    changes = robustness_prediction_changes(cv_method, feature_noise, target_noise, fold_count)
    if changes.empty:
        print("No paired individual predictions are stored for this setting.")
        return
    print(f"{CV_LABELS[cv_method]} | stored CV seeds: {sorted(changes.cv_seed.unique())} | "
          f"fold counts: {sorted(changes.fold_count.unique())} | "
          f"feature noise: {feature_noise:g}, target noise: {target_noise:g}")
    print("All prediction, change, and error values are in eV; Δ = noisy − clean prediction.")
    columns = ["observation_id", "composition", "cv_seed", "clean_prediction", "noisy_prediction",
               "prediction_change", "absolute_prediction_change", "experimental_gap",
               "clean_absolute_error", "noisy_absolute_error"]
    print("1. Largest prediction changes overall (including initially poor predictions)")
    display(changes[columns].head(12).round(3))
    accurate = changes[changes.clean_absolute_error <= clean_error_limit]
    print(f"2. Initially well predicted: clean absolute error ≤ {clean_error_limit:.2f} eV "
          f"({len(accurate)} qualifying rows); ranked by absolute prediction change")
    if accurate.empty:
        print("No materials meet this threshold. Increase the clean-error limit to explore more cases.")
    else:
        display(accurate[columns].head(12).round(3))

robust_explanation_output = interactive_output(render_robustness_explanations, {
    "cv_method": robust_explain_cv_widget, "top_n": robust_top_widget,
    "feature_noise": example_feature_noise_widget, "target_noise": example_target_noise_widget,
    "clean_error_limit": clean_error_limit_widget,
    "fold_count": robust_count_widget,
})
display(VBox([
    HBox([robust_explain_cv_widget, robust_top_widget]),
    HBox([example_feature_noise_widget, example_target_noise_widget]),
    HBox([clean_error_limit_widget, robust_count_widget]),
    robust_explanation_output,
]))


## 11. Exploring GIFTERS: Model Stability

In GIFTERS, **stability** asks whether a fitted model gives reasonably equivalent predictions after small changes to the model itself. This is deliberately different from the preceding robustness analysis:

> **Section 10 asked:** “What happens when I perturb the **data**?”  
> **Section 11 asks:** “What happens when I perturb the **fitted model**?”

Sections 1–10 compared evidence generated from many fitted models. Here you will choose one defensible configuration, train **one** local Random Forest on one reproducible train/test split, and investigate the neighborhood around that fitted model. After the button-triggered fit, neither tree reweighting nor threshold perturbation performs any additional training.

The primary evidence is movement in material-level predictions. Aggregate error remains useful, but a nearly unchanged RMSE can conceal substantial movement for particular compositions.


### 11.1 Choose and Train One Forest

Choose values from the same hyperparameter levels explored in Section 8. Moving these controls does nothing expensive. Training occurs only when you explicitly press **Train Selected Forest**.

The fit uses the notebook's fixed ensemble size and Random Forest seed plus one fixed 80/20 train/test split. Raw numeric features enter the model; `composition` is retained only alongside the held-out rows so statistically sensitive cases can be interpreted as materials.


In [ ]:
# One-model stability configuration: all important computational choices are visible here.
STABILITY_TEST_FRACTION = 0.20
STABILITY_SPLIT_SEED = 1120
STABILITY_RF_SEED = int(METADATA["rf_seed"])
WEIGHT_MC_REALIZATIONS = 200
THRESHOLD_MC_REALIZATIONS = 100
WEIGHT_SIGMA_GRID = (0.0, 0.05, 0.10, 0.20, 0.35, 0.50, 0.75)
THRESHOLD_EPSILON_GRID = (0.0, 0.005, 0.010, 0.020, 0.050)
SENSITIVE_MATERIAL_COUNT = 12
WEIGHT_MC_SEED = 11_201
THRESHOLD_MC_SEED = 11_202

STABILITY_STATE = {}
stability_refresh_widget = IntSlider(value=0, min=0, max=1_000_000)

def _depth_from_widget(value):
    return None if value in (-1, None) else int(value)

def _depth_label(value):
    return "unlimited" if value in (-1, None) else str(int(value))

def _stability_metric_summary(y_true, predictions):
    predictions = np.asarray(predictions)
    if predictions.ndim == 1:
        predictions = predictions[None, :]
    residuals = predictions - np.asarray(y_true)[None, :]
    rmse = np.sqrt(np.mean(residuals ** 2, axis=1))
    mae = np.mean(np.abs(residuals), axis=1)
    denominator = np.sum((np.asarray(y_true) - np.mean(y_true)) ** 2)
    r2 = 1.0 - np.sum(residuals ** 2, axis=1) / denominator
    return pd.DataFrame({"r2": r2, "mae": mae, "rmse": rmse})

def _node_depths(tree_structure):
    depths = np.full(tree_structure.node_count, -1, dtype=int)
    depths[0] = 0
    stack = [0]
    while stack:
        node = stack.pop()
        left = tree_structure.children_left[node]
        right = tree_structure.children_right[node]
        if left != right:
            depths[left] = depths[node] + 1
            depths[right] = depths[node] + 1
            stack.extend([left, right])
    assert np.all(depths >= 0)
    return depths

def _custom_tree_traversal(tree_cache, X_values, thresholds=None):
    """Return predictions and terminal nodes without mutating sklearn's tree."""
    thresholds = tree_cache["threshold"] if thresholds is None else thresholds
    node = np.zeros(len(X_values), dtype=np.int32)
    active = np.ones(len(X_values), dtype=bool)
    while np.any(active):
        rows = np.flatnonzero(active)
        current = node[rows]
        features = tree_cache["feature"][current]
        internal = features >= 0
        if not np.any(internal):
            active[rows] = False
            continue
        leaf_rows = rows[~internal]
        active[leaf_rows] = False
        decision_rows = rows[internal]
        decision_nodes = current[internal]
        decision_features = features[internal]
        go_left = X_values[decision_rows, decision_features] <= thresholds[decision_nodes]
        node[decision_rows] = np.where(
            go_left,
            tree_cache["children_left"][decision_nodes],
            tree_cache["children_right"][decision_nodes],
        )
    return tree_cache["value"][node], node

def _prepare_tree_caches(forest, X_train_values, X_test_values):
    """Cache topology, baseline routing, and node-local feature scales once."""
    tree_caches = []
    custom_predictions = []
    for tree_number, estimator in enumerate(forest.estimators_):
        structure = estimator.tree_
        feature = structure.feature.copy()
        threshold = structure.threshold.copy()
        internal_nodes = np.flatnonzero(feature >= 0)
        local_scale = np.zeros(structure.node_count, dtype=float)
        local_min = threshold.copy()
        local_max = threshold.copy()

        # CSC columns provide the training-row indices that reached each node.
        path_by_node = estimator.decision_path(X_train_values).tocsc()
        for node in internal_nodes:
            start, stop = path_by_node.indptr[node:node + 2]
            rows = path_by_node.indices[start:stop]
            values = X_train_values[rows, feature[node]].astype(float, copy=False)
            q25, q75 = np.percentile(values, [25.0, 75.0])
            scale = float(q75 - q25)
            if scale == 0.0:
                scale = float(np.std(values, ddof=0))
            if not np.isfinite(scale) or scale == 0.0:
                scale = 0.0
            local_scale[node] = scale
            local_min[node] = float(np.min(values))
            local_max[node] = float(np.max(values))

        cache = {
            "children_left": structure.children_left.copy(),
            "children_right": structure.children_right.copy(),
            "feature": feature,
            "threshold": threshold,
            "value": structure.value[:, 0, 0].copy(),
            "depth": _node_depths(structure),
            "local_scale": local_scale,
            "local_min": local_min,
            "local_max": local_max,
        }
        prediction, leaf = _custom_tree_traversal(cache, X_test_values)
        sklearn_prediction = estimator.predict(X_test_values)
        assert np.allclose(prediction, sklearn_prediction, rtol=0.0, atol=1e-12), (
            f"Zero-perturbation traversal mismatch in tree {tree_number}"
        )
        cache["baseline_leaf"] = leaf
        cache["baseline_prediction"] = prediction
        tree_caches.append(cache)
        custom_predictions.append(prediction)
    return tree_caches, np.asarray(custom_predictions)

def _validate_selected_hyperparameters(max_depth, min_samples_leaf, min_samples_split, max_features):
    max_depth = _depth_from_widget(max_depth)
    if max_depth is not None and max_depth < 1:
        raise ValueError("max_depth must be positive or unlimited")
    if int(min_samples_leaf) < 1 or int(min_samples_split) < 2:
        raise ValueError("Invalid minimum sample count")
    if not 0 < float(max_features) <= 1:
        raise ValueError("max_features must be in (0, 1]")
    return max_depth, int(min_samples_leaf), int(min_samples_split), float(max_features)

def _options_for_parameter(name):
    values = PARAMETER_VALUES[name]
    if name == "max_depth":
        # SelectionSlider cannot reliably use None as an internal option value.
        return [(_depth_label(value), -1 if value is None else int(value)) for value in values]
    if name == "max_features":
        return [(f"{float(value):g}", float(value)) for value in values]
    return [(str(int(value)), int(value)) for value in values]

def _preferred_or_first(options, preferred):
    available = [value for _label, value in options]
    return preferred if preferred in available else available[0]

representative = METADATA["config"]["representative_params"]
depth_options = _options_for_parameter("max_depth")
leaf_options = _options_for_parameter("min_samples_leaf")
split_options = _options_for_parameter("min_samples_split")
feature_options = _options_for_parameter("max_features")
stability_depth_widget = SelectionSlider(
    options=depth_options,
    value=_preferred_or_first(depth_options, -1 if representative[0] is None else int(representative[0])), description="max depth",
    continuous_update=False,
)
stability_leaf_widget = SelectionSlider(
    options=leaf_options,
    value=_preferred_or_first(leaf_options, int(representative[1])), description="min leaf",
    continuous_update=False,
)
stability_split_widget = SelectionSlider(
    options=split_options,
    value=_preferred_or_first(split_options, int(representative[2])), description="min split",
    continuous_update=False,
)
stability_features_widget = SelectionSlider(
    options=feature_options,
    value=_preferred_or_first(feature_options, float(representative[3])), description="max features",
    continuous_update=False,
)
train_stability_button = Button(
    description="Train Selected Forest", button_style="primary", icon="play"
)
stability_training_output = Output()

def _train_selected_stability_forest(_button=None):
    with stability_training_output:
        clear_output(wait=True)
        parameters = _validate_selected_hyperparameters(
            stability_depth_widget.value,
            stability_leaf_widget.value,
            stability_split_widget.value,
            stability_features_widget.value,
        )
        assert "composition" not in X.columns and target_column not in X.columns
        indices = np.arange(len(X))
        train_indices, test_indices = train_test_split(
            indices, test_size=STABILITY_TEST_FRACTION,
            random_state=STABILITY_SPLIT_SEED, shuffle=True,
        )
        X_train = X.iloc[train_indices].copy()
        X_test = X.iloc[test_indices].copy()
        y_train = y.iloc[train_indices].copy()
        y_test = y.iloc[test_indices].copy()
        test_compositions = compositions.iloc[test_indices].reset_index(drop=True)
        assert "composition" not in X_train.columns and "composition" not in X_test.columns

        started = time.perf_counter()
        forest = RandomForestRegressor(
            n_estimators=int(fixed_trees), max_depth=parameters[0],
            min_samples_leaf=parameters[1], min_samples_split=parameters[2],
            max_features=parameters[3], random_state=STABILITY_RF_SEED,
            n_jobs=-1,
        )
        forest.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - started
        baseline_prediction = forest.predict(X_test)
        individual_tree_predictions = np.asarray([
            estimator.predict(np.asarray(X_test, dtype=np.float32))
            for estimator in forest.estimators_
        ])
        assert np.allclose(
            individual_tree_predictions.mean(axis=0), baseline_prediction,
            rtol=0.0, atol=1e-12,
        ), "Equal tree weights must reproduce the Random Forest prediction"

        # sklearn trees route float32 inputs internally; use the same representation.
        X_train_traversal = np.asarray(X_train, dtype=np.float32)
        X_test_traversal = np.asarray(X_test, dtype=np.float32)
        cache_started = time.perf_counter()
        tree_caches, custom_tree_predictions = _prepare_tree_caches(
            forest, X_train_traversal, X_test_traversal
        )
        cache_seconds = time.perf_counter() - cache_started
        custom_forest_prediction = custom_tree_predictions.mean(axis=0)
        assert np.allclose(
            custom_forest_prediction, baseline_prediction,
            rtol=0.0, atol=1e-12,
        ), "Zero-threshold custom forest traversal must reproduce sklearn"

        baseline_metrics = _stability_metric_summary(
            y_test.to_numpy(), baseline_prediction
        ).iloc[0]
        original_thresholds = [cache["threshold"].copy() for cache in tree_caches]
        prior_training_count = STABILITY_STATE.get("training_count", 0)
        STABILITY_STATE.clear()
        STABILITY_STATE.update({
            "training_count": prior_training_count + 1,
            "parameters": parameters,
            "forest": forest,
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test,
            "test_compositions": test_compositions,
            "baseline_prediction": baseline_prediction,
            "baseline_metrics": baseline_metrics,
            "tree_predictions": individual_tree_predictions,
            "tree_caches": tree_caches,
            "original_thresholds": original_thresholds,
            "weight_results": {},
            "threshold_results": {},
            "fit_seconds": fit_seconds,
            "threshold_cache_seconds": cache_seconds,
        })

        depth_text = _depth_label(parameters[0])
        display(pd.Series({
            "max_depth": depth_text,
            "min_samples_leaf": parameters[1],
            "min_samples_split": parameters[2],
            "max_features": parameters[3],
            "n_estimators": int(fixed_trees),
            "RF random_state": STABILITY_RF_SEED,
            "split random_state": STABILITY_SPLIT_SEED,
            "training samples": len(X_train),
            "test samples": len(X_test),
            "baseline test R²": baseline_metrics.r2,
            "baseline test MAE (eV)": baseline_metrics.mae,
            "baseline test RMSE (eV)": baseline_metrics.rmse,
            "forest fit time (s)": fit_seconds,
            "threshold-cache time (s)": cache_seconds,
        }, name="selected fitted forest").to_frame())
        print(
            "Cached one fitted forest, the fixed split, compositions, baseline and "
            "tree-level predictions, terminal leaves, node depths, and local scales."
        )
    if "stability_curves_output" in globals():
        with stability_curves_output:
            clear_output(wait=True)
    stability_refresh_widget.value += 1

train_stability_button.on_click(_train_selected_stability_forest)
display(VBox([
    HBox([stability_depth_widget, stability_leaf_widget]),
    HBox([stability_split_widget, stability_features_widget]),
    train_stability_button,
    stability_training_output,
]))


### 11.2 Tree-Reweighting Stability

The ordinary forest assigns every fitted tree weight $1/T$. We perturb only those relative weights:

$$z_t\sim\mathcal{N}(0,1),\qquad u_t=\exp(\sigma_w z_t),\qquad w_t=\frac{u_t}{\sum_j u_j}.$$

At $\sigma_w=0$, the weights are exactly uniform. The effective tree count

$$N_{\mathrm{eff}}=\frac{1}{\sum_t w_t^2}$$

equals $T$ for uniform weights and falls as fewer trees dominate. Individual-tree test predictions were cached during the single fit, so every Monte Carlo ensemble below is one matrix multiplication—not another training run.


In [ ]:
tree_weight_sigma_widget = SelectionSlider(
    options=[(f"{value:.2f}", value) for value in WEIGHT_SIGMA_GRID],
    value=0.0, description="weight σ", continuous_update=False,
)

def _tree_weight_results(sigma_w):
    if not STABILITY_STATE:
        return None
    sigma_w = float(sigma_w)
    cache_key = round(sigma_w, 8)
    if cache_key in STABILITY_STATE["weight_results"]:
        return STABILITY_STATE["weight_results"][cache_key]
    started = time.perf_counter()
    tree_predictions = STABILITY_STATE["tree_predictions"]
    n_trees = tree_predictions.shape[0]
    rng = np.random.default_rng(WEIGHT_MC_SEED + int(round(10_000 * sigma_w)))
    z = rng.normal(size=(WEIGHT_MC_REALIZATIONS, n_trees))
    log_weight = sigma_w * z
    log_weight -= log_weight.max(axis=1, keepdims=True)
    weights = np.exp(log_weight)
    weights /= weights.sum(axis=1, keepdims=True)
    assert np.allclose(weights.sum(axis=1), 1.0, rtol=0.0, atol=1e-12)
    predictions = weights @ tree_predictions
    n_effective = 1.0 / np.sum(weights ** 2, axis=1)
    if sigma_w == 0.0:
        assert np.allclose(
            predictions, STABILITY_STATE["baseline_prediction"][None, :],
            rtol=0.0, atol=1e-12,
        )
        assert np.allclose(n_effective, n_trees, rtol=0.0, atol=1e-10)
    assert np.all(np.isfinite(predictions))
    results = {
        "weights": weights,
        "predictions": predictions,
        "n_effective": n_effective,
        "metrics": _stability_metric_summary(
            STABILITY_STATE["y_test"].to_numpy(), predictions
        ),
        "seconds": time.perf_counter() - started,
    }
    STABILITY_STATE["weight_results"][cache_key] = results
    return results

def _prediction_interval_table(predictions):
    baseline = STABILITY_STATE["baseline_prediction"]
    table = pd.DataFrame({
        "composition": STABILITY_STATE["test_compositions"],
        "baseline": baseline,
        "median": np.median(predictions, axis=0),
        "prediction_std": np.std(predictions, axis=0, ddof=0),
        "lower_2.5": np.percentile(predictions, 2.5, axis=0),
        "upper_97.5": np.percentile(predictions, 97.5, axis=0),
    })
    table["interval_width"] = table["upper_97.5"] - table["lower_2.5"]
    return table

def render_tree_weight_stability(sigma_w, _refresh):
    if not STABILITY_STATE:
        print("Train one selected forest in Section 11.1 first.")
        return
    results = _tree_weight_results(sigma_w)
    intervals = _prediction_interval_table(results["predictions"])
    sensitive = intervals.nlargest(SENSITIVE_MATERIAL_COUNT, "interval_width").sort_values(
        "interval_width", ascending=True
    )
    baseline_rmse = float(STABILITY_STATE["baseline_metrics"].rmse)
    representative_weights = results["weights"][0]
    representative_neff = results["n_effective"][0]
    with plt.ioff():
        figure, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    axes[0].hist(representative_weights, bins=20, color="#4c78a8", alpha=0.85)
    axes[0].axvline(1.0 / len(representative_weights), color="black", linestyle="--", label="equal weight = 1/T")
    axes[0].set(
        xlabel="tree weight", ylabel="trees",
        title=f"One perturbed ensemble\nN_eff = {representative_neff:.1f} of {len(representative_weights)}",
    )
    axes[0].legend()

    axes[1].hist(results["metrics"].rmse, bins=24, color="#54a24b", alpha=0.85)
    axes[1].axvline(baseline_rmse, color="black", linestyle="--", label=f"baseline = {baseline_rmse:.3f} eV")
    axes[1].set(xlabel="test RMSE (eV)", ylabel="Monte Carlo ensembles", title="Aggregate performance")
    axes[1].legend()

    positions = np.arange(len(sensitive))
    axes[2].hlines(positions, sensitive["lower_2.5"], sensitive["upper_97.5"], color="#4c78a8", linewidth=2, label="perturbed 95% interval")
    axes[2].scatter(sensitive["baseline"], positions, marker="D", color="#e45756", label="baseline prediction", zorder=3)
    axes[2].scatter(sensitive["median"], positions, marker="|", s=120, color="black", label="perturbed median")
    axes[2].set(
        yticks=positions, yticklabels=sensitive.composition,
        xlabel="predicted band gap (eV)", title="Most weight-sensitive materials",
    )
    axes[2].legend(fontsize=8)
    figure.suptitle(
        f"Tree reweighting | σ_w = {sigma_w:.2f} | {WEIGHT_MC_REALIZATIONS} cached-model perturbations"
    )
    display(figure)
    plt.close(figure)
    display(pd.Series({
        "median N_eff": np.median(results["n_effective"]),
        "median perturbed RMSE (eV)": np.median(results["metrics"].rmse),
        "95th percentile |prediction change| (eV)": np.percentile(
            np.abs(results["predictions"] - STABILITY_STATE["baseline_prediction"]), 95
        ),
        "Monte Carlo runtime (s)": results["seconds"],
        "additional forest fits": 0,
    }, name="tree-reweighting summary").to_frame())

tree_weight_output = interactive_output(render_tree_weight_stability, {
    "sigma_w": tree_weight_sigma_widget,
    "_refresh": stability_refresh_widget,
})
display(VBox([tree_weight_sigma_widget, tree_weight_output]))


**Interrogate the evidence:** Does the RMSE distribution remain narrow while any material-level intervals become noticeable? How quickly does $N_{\mathrm{eff}}$ fall, and do the sensitive compositions suggest a region of chemistry that deserves closer scrutiny?


### 11.3 Internal-Threshold Stability

Every internal tree node asks whether $x_j\leq\tau$. We now ask whether the fitted forest behaves similarly if those learned boundaries move slightly—without changing the input data, topology, or leaf predictions.

For a node splitting feature $j$, the perturbed threshold is

$$\tau'=\tau+\epsilon z s_{\mathrm{node},j},\qquad z\sim\mathcal{N}(0,1),$$

where $s_{\mathrm{node},j}$ is the IQR of feature $j$ among training observations that originally reached that node. A zero IQR falls back to the node-local standard deviation; if both vanish, the node is unchanged. Perturbed thresholds are clamped to that node's observed training range. The depth control restricts perturbation to upper-level decisions or allows every internal node.

Prediction uses a lightweight traversal over cached sklearn tree arrays. The original fitted forest is never copied or mutated. Zero perturbation is asserted to reproduce sklearn predictions and terminal leaves exactly.


In [ ]:
threshold_epsilon_widget = SelectionSlider(
    options=[(f"{value:.3f}", value) for value in THRESHOLD_EPSILON_GRID],
    value=0.0, description="threshold ε", continuous_update=False,
)
threshold_depth_widget = Dropdown(
    options=[("root only", 0), ("through depth 1", 1), ("through depth 2", 2),
             ("through depth 3", 3), ("through depth 4", 4), ("all internal nodes", -1)],
    value=2, description="max depth",
)

def _threshold_perturbation_results(epsilon, maximum_depth):
    if not STABILITY_STATE:
        return None
    epsilon = float(epsilon)
    maximum_depth = int(maximum_depth)
    cache_key = (round(epsilon, 8), maximum_depth)
    if cache_key in STABILITY_STATE["threshold_results"]:
        return STABILITY_STATE["threshold_results"][cache_key]
    started = time.perf_counter()
    baseline = STABILITY_STATE["baseline_prediction"]
    n_test = len(baseline)
    n_trees = len(STABILITY_STATE["tree_caches"])
    if epsilon == 0.0:
        predictions = np.repeat(baseline[None, :], THRESHOLD_MC_REALIZATIONS, axis=0)
        routing_by_material = np.zeros((THRESHOLD_MC_REALIZATIONS, n_test))
    else:
        predictions = np.empty((THRESHOLD_MC_REALIZATIONS, n_test), dtype=float)
        routing_by_material = np.empty_like(predictions)
        depth_seed = 999 if maximum_depth < 0 else maximum_depth
        rng = np.random.default_rng(
            THRESHOLD_MC_SEED + int(round(epsilon * 1_000_000)) + 10_000 * depth_seed
        )
        X_test_values = np.asarray(STABILITY_STATE["X_test"], dtype=np.float32)
        for realization in range(THRESHOLD_MC_REALIZATIONS):
            prediction_sum = np.zeros(n_test, dtype=float)
            changed_count = np.zeros(n_test, dtype=float)
            for tree_cache in STABILITY_STATE["tree_caches"]:
                eligible = (
                    (tree_cache["feature"] >= 0)
                    & (tree_cache["local_scale"] > 0)
                    & ((maximum_depth < 0) | (tree_cache["depth"] <= maximum_depth))
                )
                perturbed_threshold = tree_cache["threshold"].copy()
                nodes = np.flatnonzero(eligible)
                if len(nodes):
                    perturbed_threshold[nodes] += (
                        epsilon * rng.normal(size=len(nodes)) * tree_cache["local_scale"][nodes]
                    )
                    perturbed_threshold[nodes] = np.clip(
                        perturbed_threshold[nodes],
                        tree_cache["local_min"][nodes], tree_cache["local_max"][nodes],
                    )
                tree_prediction, terminal_leaf = _custom_tree_traversal(
                    tree_cache, X_test_values, perturbed_threshold
                )
                prediction_sum += tree_prediction
                changed_count += terminal_leaf != tree_cache["baseline_leaf"]
            predictions[realization] = prediction_sum / n_trees
            routing_by_material[realization] = changed_count / n_trees

    assert np.all(np.isfinite(predictions))
    if epsilon == 0.0:
        assert np.allclose(predictions, baseline[None, :], rtol=0.0, atol=1e-12)
        assert np.count_nonzero(routing_by_material) == 0
    for estimator, original in zip(
        STABILITY_STATE["forest"].estimators_, STABILITY_STATE["original_thresholds"]
    ):
        assert np.array_equal(estimator.tree_.threshold, original), (
            "Threshold perturbation must never mutate the fitted sklearn forest"
        )
    results = {
        "predictions": predictions,
        "routing_by_material": routing_by_material,
        "overall_routing_change": routing_by_material.mean(axis=1),
        "metrics": _stability_metric_summary(
            STABILITY_STATE["y_test"].to_numpy(), predictions
        ),
        "seconds": time.perf_counter() - started,
    }
    STABILITY_STATE["threshold_results"][cache_key] = results
    return results

def render_threshold_stability(epsilon, maximum_depth, _refresh):
    if not STABILITY_STATE:
        print("Train one selected forest in Section 11.1 first.")
        return
    results = _threshold_perturbation_results(epsilon, maximum_depth)
    intervals = _prediction_interval_table(results["predictions"])
    intervals["mean_routing_change"] = results["routing_by_material"].mean(axis=0)
    sensitive = intervals.nlargest(SENSITIVE_MATERIAL_COUNT, "interval_width").sort_values(
        "interval_width", ascending=True
    )
    baseline_rmse = float(STABILITY_STATE["baseline_metrics"].rmse)
    depth_label = "all" if maximum_depth < 0 else str(maximum_depth)
    with plt.ioff():
        figure, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    axes[0].hist(results["metrics"].rmse, bins=24, color="#54a24b", alpha=0.85)
    axes[0].axvline(baseline_rmse, color="black", linestyle="--", label=f"baseline = {baseline_rmse:.3f} eV")
    axes[0].set(xlabel="test RMSE (eV)", ylabel="Monte Carlo forests", title="Aggregate performance")
    axes[0].legend()

    positions = np.arange(len(sensitive))
    axes[1].hlines(positions, sensitive["lower_2.5"], sensitive["upper_97.5"], color="#4c78a8", linewidth=2, label="perturbed 95% interval")
    axes[1].scatter(sensitive["baseline"], positions, marker="D", color="#e45756", label="baseline prediction", zorder=3)
    axes[1].scatter(sensitive["median"], positions, marker="|", s=120, color="black", label="perturbed median")
    axes[1].set(
        yticks=positions, yticklabels=sensitive.composition,
        xlabel="predicted band gap (eV)", title="Most threshold-sensitive materials",
    )
    axes[1].legend(fontsize=8)

    mean_material_routing = results["routing_by_material"].mean(axis=0)
    axes[2].hist(mean_material_routing, bins=24, color="#f58518", alpha=0.85)
    axes[2].axvline(mean_material_routing.mean(), color="black", linestyle="--", label="test-set mean")
    axes[2].set(
        xlabel="fraction of trees reaching a different leaf",
        ylabel="test materials", title="Material-level routing sensitivity",
    )
    axes[2].legend()
    figure.suptitle(
        f"Threshold perturbation | ε = {epsilon:.3f} × node-local scale | max depth = {depth_label} | "
        f"{THRESHOLD_MC_REALIZATIONS} perturbations"
    )
    display(figure)
    plt.close(figure)
    display(pd.Series({
        "median fraction of changed tree–sample paths": np.median(results["overall_routing_change"]),
        "95th percentile changed-path fraction": np.percentile(results["overall_routing_change"], 95),
        "median perturbed RMSE (eV)": np.median(results["metrics"].rmse),
        "95th percentile |prediction change| (eV)": np.percentile(
            np.abs(results["predictions"] - STABILITY_STATE["baseline_prediction"]), 95
        ),
        "Monte Carlo runtime (s)": results["seconds"],
        "additional forest fits": 0,
    }, name="threshold-perturbation summary").to_frame())

threshold_stability_output = interactive_output(render_threshold_stability, {
    "epsilon": threshold_epsilon_widget,
    "maximum_depth": threshold_depth_widget,
    "_refresh": stability_refresh_widget,
})
display(VBox([
    HBox([threshold_epsilon_widget, threshold_depth_widget]),
    threshold_stability_output,
]))


**Interrogate the evidence:** Follow the causal chain from threshold movement to changed routing, changed leaf values, and changed ensemble prediction. Do upper-level changes matter more than perturbing all fine-scale decisions? Are routing-sensitive materials also prediction-sensitive?


### 11.4 Stability Curves

Detailed views show one perturbation setting at a time. The summary below sweeps the predefined tree-weight grid and threshold-strength grid at the selected maximum depth. It emphasizes the median and 95th percentile of absolute prediction movement across test materials and Monte Carlo realizations; the lower row retains median RMSE change as secondary evidence.

Computing the curves performs no fitting. Tree reweighting is essentially instantaneous; uncached threshold settings require custom traversal and may take several seconds each.


In [ ]:
compute_stability_curves_button = Button(
    description="Compute Stability Curves", button_style="info", icon="line-chart"
)
stability_curves_output = Output()

def _stability_curve_rows(results_by_magnitude, mechanism):
    baseline_prediction = STABILITY_STATE["baseline_prediction"]
    baseline_rmse = float(STABILITY_STATE["baseline_metrics"].rmse)
    rows = []
    for magnitude, results in results_by_magnitude:
        movement = np.abs(results["predictions"] - baseline_prediction[None, :])
        rows.append({
            "mechanism": mechanism,
            "magnitude": magnitude,
            "median_absolute_prediction_change": np.median(movement),
            "p95_absolute_prediction_change": np.percentile(movement, 95),
            "delta_rmse": np.median(results["metrics"].rmse) - baseline_rmse,
            "runtime_seconds": results["seconds"],
        })
    return rows

def _compute_stability_curves(_button=None):
    with stability_curves_output:
        clear_output(wait=True)
        if not STABILITY_STATE:
            print("Train one selected forest in Section 11.1 first.")
            return
        started = time.perf_counter()
        maximum_depth = int(threshold_depth_widget.value)
        weight_results = [(sigma, _tree_weight_results(sigma)) for sigma in WEIGHT_SIGMA_GRID]
        threshold_results = [
            (epsilon, _threshold_perturbation_results(epsilon, maximum_depth))
            for epsilon in THRESHOLD_EPSILON_GRID
        ]
        curve_data = pd.DataFrame(
            _stability_curve_rows(weight_results, "tree reweighting")
            + _stability_curve_rows(threshold_results, "threshold movement")
        )
        with plt.ioff():
            figure, axes = plt.subplots(
                2, 2, figsize=(14, 9), sharey="row", constrained_layout=True
            )
        specifications = [
            ("tree reweighting", r"tree-weight perturbation $\sigma_w$", axes[0, 0], axes[1, 0], "#4c78a8"),
            ("threshold movement", r"threshold perturbation $\epsilon$", axes[0, 1], axes[1, 1], "#f58518"),
        ]
        for mechanism, x_label, movement_axis, rmse_axis, color in specifications:
            current = curve_data[curve_data.mechanism == mechanism]
            movement_axis.plot(
                current.magnitude, current.median_absolute_prediction_change,
                marker="o", color=color, label="median |Δ prediction|",
            )
            movement_axis.plot(
                current.magnitude, current.p95_absolute_prediction_change,
                marker="s", linestyle="--", color="#e45756", label="95th percentile |Δ prediction|",
            )
            movement_axis.axhline(0, color="0.5", linewidth=0.8)
            movement_axis.set(xlabel=x_label, ylabel="prediction movement (eV)", title=mechanism.title())
            movement_axis.legend()
            movement_axis.grid(color="0.92")
            rmse_axis.plot(current.magnitude, current.delta_rmse, marker="o", color=color)
            rmse_axis.axhline(0, color="black", linestyle="--", linewidth=0.9)
            rmse_axis.set(xlabel=x_label, ylabel="median ΔRMSE (eV)")
            rmse_axis.grid(color="0.92")
        depth_label = "all" if maximum_depth < 0 else str(maximum_depth)
        figure.suptitle(
            f"Local stability of one fitted forest | threshold maximum depth = {depth_label}"
        )
        display(figure)
        plt.close(figure)
        total_seconds = time.perf_counter() - started
        display(curve_data.round(6))
        print(
            f"Curve computation took {total_seconds:.2f} s; all points used the same fitted forest "
            "and zero additional fits. Cached settings return immediately."
        )

compute_stability_curves_button.on_click(_compute_stability_curves)
display(VBox([compute_stability_curves_button, stability_curves_output]))


### 11.5 Pedagogical Synthesis

Use the linked evidence to build a qualified stability claim rather than labeling the model simply “stable” or “unstable.”

- Does modest tree reweighting materially alter predictions?
- How rapidly does $N_{\mathrm{eff}}$ fall as tree weights become uneven?
- Can aggregate RMSE remain nearly constant while individual predictions move?
- Which materials are especially sensitive to tree weighting?
- How much threshold perturbation is needed before routing starts changing?
- Are upper-level decisions more consequential than deep decisions?
- Do the same compositions appear sensitive to both perturbation mechanisms?
- Is this fitted model more stable to ensemble aggregation changes or to internal boundary changes?
- Does materials-science knowledge suggest why particular compositions are unstable?

> **A stable aggregate error metric does not guarantee stable individual predictions.**

These experiments characterize a local neighborhood around one fitted forest, one split, and one chosen configuration. They are evidence about that fitted model—not a universal guarantee about every Random Forest trained on this dataset.


## Adding Experiments Without Replacing Saved Results

The generator now checks individual experiment keys and computes only missing results. A seed's predictions and metrics are saved together; rerunning after interruption resumes at the next incomplete experiment. Original metadata stays intact; `generation_history` and `experiment_provenance` record new configurations and library versions. Changing libraries does not invalidate old results; comparisons across versions should be interpreted with that provenance in mind.

From the repository root, extend Section 6 using all CPU cores for each forest:

```bash
python demonstrations/generate_rfr_precomputed.py --studies model_selection --jobs -1 --yes
```

Use `--config experiments.json` to override selected settings; existing database settings are otherwise inherited. For example:

```json
{
  "selection_outer_seeds": [0, 1, 2, 3, 4, 5],
  "selection_leaf_sizes": [1, 2, 4, 8, 16, 32],
  "selection_split_sizes": [2, 4, 8, 16, 32, 64],
  "validation_seeds": [30, 31],
  "loco_counts": [5, 10, 15],
  "min_samples_leaf_grid": [1, 2, 4, 8]
}
```

```bash
python demonstrations/generate_rfr_precomputed.py --config experiments.json --studies validation hyperparameters clusters --yes
```

The JSON specifies the combinations requested in this run, not deletions: previously saved rows remain. Select from `clusters`, `convergence`, `model_selection`, `validation`, `hyperparameters`, `importance`, and `robustness`. Increasing `validation_seeds` also adds KMeans partition seeds for LOCO. To add an importance seed/count, request another `importance_cv_seed`/`importance_fold_count`; robustness uses `robustness_seeds` and `robustness_fold_count`. New robustness runs retain individual predictions for every seed.

Candidate-set changes in Section 6 define a new study and require new nested fits. Changes to the dataset, fixed tree count, representative model, inner/outer selection fold counts, or permutation repeat count require a separate `--database`, so incompatible definitions cannot silently share result keys. `--overwrite` is still an explicit destructive reset; it is not needed to extend grids or seeds. New databases default to smoke settings; use `--mode full` for full settings. Restart/rerun notebook setup and study cells after generation to refresh cached query results.
